# Chapter 3 · Lab 1 — Diagnose the performance gap

Read [background.md](../background.md), then run cells top to bottom in a fresh kernel.
Prerequisites: Chapter 1's validated Qwen3/cache and checkpoint loader; Chapter 2's
FLOP/byte ledger, roofline, CUDA event timing and frozen predictions. This lab adds
reusable [profiling.py](profiling.py) and [study.py](study.py), used again by Labs 2/3.
It requires the pinned local **8B and 32B checkpoints**, BF16-capable GB10, and the
[course environment](../../../shared/SETUP.md). Use `uv sync --extra kernels`.
CPU oracle results are not model GPU measurements.

Follow **prediction → implementation → correctness → measurement → explanation**.
The complete driver is runnable; exercises change its capture scope and interpret
actual evidence. Loading, validation and compilation stay outside captured calls.

In [1]:
from pathlib import Path
import importlib, json, os, sys, time
from uuid import uuid4
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'pyproject.toml').is_file())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
# Keep compiler disk caches from different DSL versions separate. In-memory JIT reuse remains enabled.
os.environ.setdefault('CUTE_DSL_DISABLE_FILE_CACHING', '1')
from IPython.display import display, Markdown, Image
from shared import performance as perf
study = importlib.import_module('chapters.03_kernels.code.study')
plots = importlib.import_module('chapters.03_kernels.code.plots')
RUN_GPU = True
MODEL_KEYS = ['8b', '32b']
EXTERNAL_PROFILERS = True
# Set P03_REPLAY_DIR only to inspect an existing run. Replay never creates measurements.
REPLAY_DIR = os.environ.get('P03_REPLAY_DIR')

/home/huangruoyu/workspace/inference-engineer-study-hall/courses/llm-inference-performance-modeling/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Freeze the prediction and identify the forward boundary

For each model, use batches 1 and 4, prompt length 2048, empty cache before prefill,
and prefix length 2048 before the first decode append. Derive the shapes of QK,
expanded KV and SwiGLU intermediates. Reproduce 32B's rectangular Q projection:
`[B*T,5120] @ [5120,8192]`. Verify dimensions against local config files.

The next cell shows Chapter 2 predictions. `study.execute` writes all per-step
predictions and source hashes **before** loading models or taking measurements.
The assumed dense BF16 ceiling is 125 TFLOP/s; 273 GB/s is the labeled hardware
bandwidth. These ideal bounds exclude launch, allocation and extra attention traffic.
Local TTFT includes prefill plus greedy token selection; events bracket only forward.

In [2]:
for key in MODEL_KEYS:
    print(key, perf.MODELS[key])
    display(perf.predict_tradeoffs(perf.MODELS[key], [1,4], 2048, study.hardware(), 'bfloat16'))
# Exercise: write predicted bottleneck, bytes saved, and latency reduction before the run.
hypotheses = [dict(operation=name, evidence_to_seek=evidence, predicted_local_speedup=2.0,
                   rationale='Replace this illustrative 2x assumption with your derivation')
              for name,evidence in [('attention','score/softmax allocations and repeat_interleave'),
                                    ('normalize','square, mean, rsqrt, scale launches'),
                                    ('activation','SiLU temporary and multiply'),
                                    ('cache','cat allocation/copy'),('host','gaps between short launches')]]
display(hypotheses)

8b ModelDimensions(layers=36, hidden=4096, intermediate=12288, query_heads=32, kv_heads=8, head_dim=128, vocab=151936)


[{'batch': 1,
  'context': 2048,
  'phase': 'prefill',
  'prefix_tokens': 0,
  'attended_positions': 2048,
  'flops': 29688662589440,
  'bytes_proxy': 15472659200,
  'compute_ms': 237.50930071552,
  'memory_ms': 56.67640732600733,
  'limiting_term': 'compute',
  'latency_ms': 237.50930071552,
  'total_tokens_per_second': 8622.820217272332,
  'token_kind': 'input',
  'interactivity_tokens_per_second': None},
 {'batch': 1,
  'context': 2048,
  'phase': 'decode',
  'prefix_tokens': 2048,
  'attended_positions': 2049,
  'flops': 16344743936,
  'bytes_proxy': 15439268608,
  'compute_ms': 0.130757951488,
  'memory_ms': 56.554097465201465,
  'limiting_term': 'memory',
  'latency_ms': 56.554097465201465,
  'total_tokens_per_second': 17.682184754434708,
  'token_kind': 'output',
  'interactivity_tokens_per_second': 17.682184754434708},
 {'batch': 4,
  'context': 2048,
  'phase': 'prefill',
  'prefix_tokens': 0,
  'attended_positions': 2048,
  'flops': 118754650357760,
  'bytes_proxy': 164802037

32b ModelDimensions(layers=64, hidden=5120, intermediate=25600, query_heads=64, kv_heads=8, head_dim=128, vocab=151936)


[{'batch': 1,
  'context': 2048,
  'phase': 'prefill',
  'prefix_tokens': 0,
  'attended_positions': 2048,
  'flops': 132219976548352,
  'bytes_proxy': 64547539712,
  'compute_ms': 1057.759812386816,
  'memory_ms': 236.4378744029304,
  'limiting_term': 'compute',
  'latency_ms': 1057.7598123868158,
  'total_tokens_per_second': 1936.1673378181433,
  'token_kind': 'input',
  'interactivity_tokens_per_second': None},
 {'batch': 1,
  'context': 2048,
  'phase': 'decode',
  'prefix_tokens': 2048,
  'attended_positions': 2049,
  'flops': 68264132608,
  'bytes_proxy': 64505879296,
  'compute_ms': 0.546113060864,
  'memory_ms': 236.28527214652016,
  'limiting_term': 'memory',
  'latency_ms': 236.28527214652016,
  'total_tokens_per_second': 4.232172369084017,
  'token_kind': 'output',
  'interactivity_tokens_per_second': 4.232172369084017},
 {'batch': 4,
  'context': 2048,
  'phase': 'prefill',
  'prefix_tokens': 0,
  'attended_positions': 2048,
  'flops': 528879906193408,
  'bytes_proxy': 6628

[{'operation': 'attention',
  'evidence_to_seek': 'score/softmax allocations and repeat_interleave',
  'predicted_local_speedup': 2.0,
  'rationale': 'Replace this illustrative 2x assumption with your derivation'},
 {'operation': 'normalize',
  'evidence_to_seek': 'square, mean, rsqrt, scale launches',
  'predicted_local_speedup': 2.0,
  'rationale': 'Replace this illustrative 2x assumption with your derivation'},
 {'operation': 'activation',
  'evidence_to_seek': 'SiLU temporary and multiply',
  'predicted_local_speedup': 2.0,
  'rationale': 'Replace this illustrative 2x assumption with your derivation'},
 {'operation': 'cache',
  'evidence_to_seek': 'cat allocation/copy',
  'predicted_local_speedup': 2.0,
  'rationale': 'Replace this illustrative 2x assumption with your derivation'},
 {'operation': 'host',
  'evidence_to_seek': 'gaps between short launches',
  'predicted_local_speedup': 2.0,
  'rationale': 'Replace this illustrative 2x assumption with your derivation'}]

## 2. Learn the tools in order

1. **[PyTorch Profiler](https://docs.pytorch.org/docs/stable/profiler.html)**:
   correlate Python operators, shapes, allocated tensors and CUDA kernels. Inspect
   `operators.csv`, `operators.txt`, `trace.json`, and `kernel_summary.json` for every
   model/batch/phase. Open traces in a compatible trace viewer. Locate
   `repeat_interleave`, matmul, softmax, norm/activation, and concatenation.
2. **[Nsight Systems](https://docs.nvidia.com/nsight-systems/UserGuide/)**:
   follow `prefill_B1_S2048`, `decode_B1_S2048`, and batch-4 NVTX ranges. Read
   kernel launches, host dispatch, synchronization, copies and idle gaps. Open
   `report.nsys-rep`; inspect `summary.csv` (`cuda_gpu_kern_sum`, `cuda_api_sum`,
   `nvtx_sum`). Separate host overhead from kernel execution.
3. **[Nsight Compute](https://docs.nvidia.com/nsight-compute/ProfilingGuide/)**:
   select the attention/RMS/softmax kernels identified above. Inspect DRAM bytes,
   occupancy, registers and compute activity. `command.log` saves exact commands;
   `status.json` preserves counter permission failures. On the preparation GB10,
   `ERR_NVGPUCTRPERM` prevents this evidence. Continue other measurements and mark
   counters incomplete; administrator access is an external prerequisite.

Each tool runs separately. `cudaProfilerStart/Stop` scopes external capture after
all cases warm; NVTX identifies each case. No profiler timings enter `results.csv`.

In [3]:
# Driver interface: model / implementation / batch / context / phase / profiler.
import subprocess
print(subprocess.check_output([sys.executable, str(ROOT/'chapters/03_kernels/code/study.py'), '--help'], text=True))

usage: study.py [-h] --out OUT [--model {8b,32b}]
                [--implementation {baseline,optimized,fusion_only,attention_only}]
                [--batch BATCH] [--context CONTEXT] [--phase {prefill,decode}]
                [--profiler {torch,nsys,ncu}] [--no-external]
                [--capture-grid CAPTURE_GRID]
                {baseline,optimized,sweep,capture}

Reproducible real-checkpoint profiling and matched Chapter 2 sweeps. Run
--help; notebooks call the same functions. No CPU timing fallback.

positional arguments:
  {baseline,optimized,sweep,capture}

options:
  -h, --help            show this help message and exit
  --out OUT
  --model {8b,32b}
  --implementation {baseline,optimized,fusion_only,attention_only}
  --batch BATCH
  --context CONTEXT
  --phase {prefill,decode}
  --profiler {torch,nsys,ncu}
  --no-external
  --capture-grid CAPTURE_GRID



## 3. Establish unprofiled baselines, validate, then capture

The driver loads models sequentially, checks full-prefix versus cached/chunked
logits and cache ownership, then runs two complete warmups and three repeats,
each with eight decode calls. Prompt IDs, forced continuation, revisions and
precision are saved. Greedy selection remains inside the wall timing boundary,
but both implementations will receive identical forced history for controlled comparison.

**Implementation exercise:** inspect `prepare_call`. Explain why repeatedly appending
into a shared mutable cache would invalidate its fixed-prefix decode capture.
Add a new `context=129` capture to exercise a partial tile, preserving the original
2048 cases. Predict which intermediates scale quadratically and which linearly.

In [4]:
if RUN_GPU:
    if REPLAY_DIR:
        RUN_DIR = Path(REPLAY_DIR).resolve()
        print('Replaying saved evidence; no new measurements:', RUN_DIR)
        assert (RUN_DIR/'completion.json').is_file()
    else:
        RUN_DIR = ROOT/'results'/('p03-baseline-' + time.strftime('%Y%m%d-%H%M%S') + '-' + uuid4().hex[:6])
        study.execute(RUN_DIR, 'baseline', MODEL_KEYS, external=EXTERNAL_PROFILERS)
    raw = plots.read_rows(RUN_DIR/'results.csv')
    summary = perf.summarize_model_rows(raw)
    display(Markdown(f'Artifacts: `{RUN_DIR}` · {len(raw)} unprofiled observations'))
    display(json.loads((RUN_DIR/'status.json').read_text()))
else:
    print('GPU/checkpoint work unmeasured. Enable RUN_GPU after setup.')

Loading 8b baseline
Measure 8b baseline 1 2048


USDT:2026-09-20 12:01:49 189581:189581 SyncActivityProfilerHandler.cpp:39] profiler_start
USDT:2026-09-20 12:01:50 189581:189581 SyncActivityProfilerHandler.cpp:46] profiler_stop
USDT:2026-09-20 12:01:53 189581:189581 SyncActivityProfilerHandler.cpp:39] profiler_start
USDT:2026-09-20 12:01:53 189581:189581 SyncActivityProfilerHandler.cpp:46] profiler_stop


Measure 8b baseline 4 2048


USDT:2026-09-20 12:02:51 189581:189581 SyncActivityProfilerHandler.cpp:39] profiler_start
USDT:2026-09-20 12:02:58 189581:189581 SyncActivityProfilerHandler.cpp:46] profiler_stop
USDT:2026-09-20 12:03:06 189581:189581 SyncActivityProfilerHandler.cpp:39] profiler_start
USDT:2026-09-20 12:03:06 189581:189581 SyncActivityProfilerHandler.cpp:46] profiler_stop


Loading 32b baseline
Measure 32b baseline 1 2048


...

Measure 32b baseline 4 2048


USDT:2026-09-20 12:12:47 189581:189581 SyncActivityProfilerHandler.cpp:39] profiler_start
USDT:2026-09-20 12:13:14 189581:189581 SyncActivityProfilerHandler.cpp:46] profiler_stop
USDT:2026-09-20 12:13:44 189581:189581 SyncActivityProfilerHandler.cpp:39] profiler_start
USDT:2026-09-20 12:13:44 189581:189581 SyncActivityProfilerHandler.cpp:46] profiler_stop


Profile nsys 8b baseline [{'batch': 1, 'context': 2048, 'phase': 'prefill'}, {'batch': 1, 'context': 2048, 'phase': 'decode'}, {'batch': 4, 'context': 2048, 'phase': 'prefill'}, {'batch': 4, 'context': 2048, 'phase': 'decode'}]
Profile ncu 8b baseline [{'batch': 1, 'context': 2048, 'phase': 'prefill'}, {'batch': 1, 'context': 2048, 'phase': 'decode'}, {'batch': 4, 'context': 2048, 'phase': 'prefill'}, {'batch': 4, 'context': 2048, 'phase': 'decode'}]
Profile nsys 32b baseline [{'batch': 1, 'context': 2048, 'phase': 'prefill'}, {'batch': 1, 'context': 2048, 'phase': 'decode'}, {'batch': 4, 'context': 2048, 'phase': 'prefill'}, {'batch': 4, 'context': 2048, 'phase': 'decode'}]
Profile ncu 32b baseline [{'batch': 1, 'context': 2048, 'phase': 'prefill'}, {'batch': 1, 'context': 2048, 'phase': 'decode'}, {'batch': 4, 'context': 2048, 'phase': 'prefill'}, {'batch': 4, 'context': 2048, 'phase': 'decode'}]


Artifacts: `/home/huangruoyu/workspace/inference-engineer-study-hall/courses/llm-inference-performance-modeling/results/p03-baseline-20260920-120013-42356d` · 108 unprofiled observations

[{'model': '8b',
  'implementation': 'baseline',
  'batch': 1,
  'prompt': 2048,
  'status': 'measured'},
 {'model': '8b',
  'implementation': 'baseline',
  'batch': 4,
  'prompt': 2048,
  'status': 'measured'},
 {'model': '32b',
  'implementation': 'baseline',
  'batch': 1,
  'prompt': 2048,
  'status': 'measured'},
 {'model': '32b',
  'implementation': 'baseline',
  'batch': 4,
  'prompt': 2048,
  'status': 'measured'}]

## 4. Rank mechanisms using evidence

For each case record a trace/operator range, proposed mechanism, expected
traffic/launch reduction, baseline fraction and predicted integrated benefit.
`f` must refer to one exclusive scope; do not add inclusive parent and child time.
Use `1 / ((1-f) + f/s + adapter_ms/baseline_ms)`. Profile-derived fractions are
perturbed estimates, so compare the prediction against independent ordinary timing.
Dynamic cache concatenation and host overhead remain for later chapters.

In [5]:
ranked_optimizations = [dict(rank=i+1, operation=h['operation'], evidence=None,
    traffic_or_launch_reduction=None, baseline_time_fraction=None,
    predicted_local_speedup=h['predicted_local_speedup'], predicted_integrated_speedup=None)
    for i,h in enumerate(hypotheses)]
display(ranked_optimizations)
# Fill from actual traces before Lab 2. None means not yet supported, never zero cost.
if RUN_GPU:
    perf.write_json(RUN_DIR/'student_ranked_optimizations.json', ranked_optimizations)
    for key in MODEL_KEYS:
        for batch in [1,4]:
            for phase in ['prefill','decode']:
                path = RUN_DIR/f'{key}-baseline-B{batch}-S2048-{phase}'/'torch/kernel_summary.json'
                if path.exists():
                    print(key, batch, phase, json.loads(path.read_text())['count'], 'CUDA kernels')
if RUN_GPU:
    display(json.loads((RUN_DIR/'ranked_optimizations.json').read_text()))


[{'rank': 1,
  'operation': 'attention',
  'evidence': None,
  'traffic_or_launch_reduction': None,
  'baseline_time_fraction': None,
  'predicted_local_speedup': 2.0,
  'predicted_integrated_speedup': None},
 {'rank': 2,
  'operation': 'normalize',
  'evidence': None,
  'traffic_or_launch_reduction': None,
  'baseline_time_fraction': None,
  'predicted_local_speedup': 2.0,
  'predicted_integrated_speedup': None},
 {'rank': 3,
  'operation': 'activation',
  'evidence': None,
  'traffic_or_launch_reduction': None,
  'baseline_time_fraction': None,
  'predicted_local_speedup': 2.0,
  'predicted_integrated_speedup': None},
 {'rank': 4,
  'operation': 'cache',
  'evidence': None,
  'traffic_or_launch_reduction': None,
  'baseline_time_fraction': None,
  'predicted_local_speedup': 2.0,
  'predicted_integrated_speedup': None},
 {'rank': 5,
  'operation': 'host',
  'evidence': None,
  'traffic_or_launch_reduction': None,
  'baseline_time_fraction': None,
  'predicted_local_speedup': 2.0,
  'p

8b 1 prefill 3430 CUDA kernels
8b 1 decode 3466 CUDA kernels
8b 4 prefill 3466 CUDA kernels
8b 4 decode 3538 CUDA kernels
32b 1 prefill 6090 CUDA kernels
32b 1 decode 6154 CUDA kernels
32b 4 prefill 6282 CUDA kernels
32b 4 decode 6282 CUDA kernels


[{'model': '32b',
  'batch': 1,
  'prompt': 2048,
  'phase': 'decode',
  'operation': 'attention',
  'evidence': '32b-baseline-B1-S2048-decode/torch/trace.json',
  'mechanism': 'CuTe causal GQA: no global scores/probabilities or expanded KV',
  'expected_logical_byte_reduction_per_layer': 69239808,
  'bytes_scope': 'illustrative intermediate write/read ledger; not measured traffic',
  'baseline_time_fraction': 0.2707746803723693,
  'fraction_scope': 'attributed kernel time / instrumented CUDA span; idle separate',
  'baseline_kernel_count': 896,
  'predicted_local_speedup': 2.0,
  'local_speedup_basis': 'assumed hypothesis, frozen before optimized run',
  'predicted_integrated_speedup': 1.1565872748328008},
 {'model': '32b',
  'batch': 1,
  'prompt': 2048,
  'phase': 'decode',
  'operation': 'normalize_rotary',
  'evidence': '32b-baseline-B1-S2048-decode/torch/trace.json',
  'mechanism': 'Fuse Q/K head norm with absolute-position rotate-half',
  'expected_logical_byte_reduction_per_lay

## Completion and explanation

Complete both models × both batches × both phases. Retain frozen predictions,
raw unprofiled repeats, correctness, all PyTorch traces/operator tables, Nsight
Systems reports/summaries, and targeted Nsight Compute reports or failure logs.
Explain five candidates: materialized scores, repeated KV heads, normalization/
activation launches, cache concatenation, and host overhead. Rank them using
measured evidence, not intuition. Reproduce one FLOP and one traffic calculation
without code and defend which cost the proposed fusion cannot remove.
Continue to [Lab 2](lab2.ipynb). Counter access can remain explicitly incomplete.